# 13 — Data Leakage Detection (Lesson)

## Problem Definition
Detect hidden leakage and shift before leaderboard submission.

## Required Prior Knowledge
- Notebook 12 cross-validation estimators and OOF protocol.
- Probability densities and cumulative distribution functions.

## New Concepts Introduced
- Distribution-shift formalism $P_{train}(X)\ne P_{test}(X)$.
- KL divergence derivation.
- Kolmogorov-Smirnov statistic derivation.
- Adversarial validation for covariate shift.
- Synthetic leakage stress testing.

## Formal Definition
Distribution shift:
$$
P_{\text{train}}(X)\neq P_{\text{test}}(X).
$$
KL divergence:
$$
D_{\mathrm{KL}}(P\parallel Q)=\int p(x)\log\frac{p(x)}{q(x)}\,dx.
$$
Empirical KS statistic for two samples:
$$
D_{n,m}=\sup_x\left|F_n(x)-G_m(x)\right|.
$$
Adversarial validation solves:
$$
\text{AUC}\big(h(x),s\big),\quad s\in\{0,1\},
$$
where $s=0$ indicates train and $s=1$ indicates test-like split.
High AUC implies strong covariate distinguishability.

## Variables and Assumptions
- Scope: Data Leakage Detection targets the objective `Detect hidden leakage and shift before leaderboard submission.`.
- Data are indexed by $i\in\{1,\dots,n\}$ with features $x_i\in\mathbb{R}^d$ and target $y_i$.
- Model parameters are represented by $\theta$, and predictions are $\hat{y}_i=f_\theta(x_i)$.
- Any preprocessing statistics are fit on training partitions only and reused on validation/test partitions.
- Split protocol (KFold/Group/TimeSeries) must match the data-generating assumptions for IID, grouped, or temporal samples.
- Reported metrics are empirical estimates with finite-sample variance; interpretation must include uncertainty.

## Symbol-by-Symbol Explanation
| Symbol | Meaning |
|---|---|
| $x_i$ | feature vector for sample $i$ |
| $y_i$ | target for sample $i$ |
| $f_\theta$ | model parameterized by $\theta$ |
| $L(\cdot,\cdot)$ | per-sample loss function |
| $n$ | number of samples |
| $d$ | raw feature dimension |
| $p$ | transformed feature dimension / polynomial degree context |
| $K$ | number of folds / partitions |
| $V_k$ | validation index set for fold $k$ |
| $\lambda,\alpha,\tau$ | regularization/smoothing/confidence hyperparameters |

## Zero-Skip Derivation
1. KL from expected log density ratio:
   $$
   D_{\mathrm{KL}}(P\parallel Q)=\mathbb{E}_{x\sim P}\left[\log p(x)-\log q(x)\right].
   $$
2. If $p=q$, integrand is zero everywhere so KL is zero.
3. KS compares empirical CDFs:
   $$
   F_n(x)=\frac{1}{n}\sum_{i=1}^n\mathbf{1}\{X_i\le x\},\quad
   G_m(x)=\frac{1}{m}\sum_{j=1}^m\mathbf{1}\{Y_j\le x\}.
   $$
4. Maximum absolute vertical distance gives $D_{n,m}$.
5. Leakage can be viewed as hidden variable $z$ carrying label information into features:
   $$
   I(Y;Z\mid \text{split})>0
   $$
   where $Z$ should be unavailable at prediction time.

## Explicit Logical Transitions
0. Context anchor: this notebook focuses on `Data Leakage Detection` and objective `Detect hidden leakage and shift before leaderboard submission.`.
1. Start from the formal objective and identify estimators/transformations introduced in this notebook.
2. Map each estimator term to computable quantities under train/validation split constraints.
3. Show why each derivation step is valid (algebraic identity, estimator definition, or probabilistic assumption).
4. Convert the derivation into an implementation protocol with explicit leakage controls.
5. Validate with synthetic and real-data experiments, then interpret failure conditions.

## Intuition
Shift metrics and adversarial validation are early warning systems: if train and test are separable, naive CV may be over-optimistic.

## Mapping from Math to Implementation
- KL and KS are computed on selected feature marginals.
- Adversarial validation trains a classifier to separate train/test domains.
- Synthetic leakage introduces a near-target proxy to verify detection diagnostics.

In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import torch
import torch.nn as nn

from sklearn.datasets import load_diabetes, load_breast_cancer, load_wine, fetch_california_housing
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, roc_auc_score, accuracy_score
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, HistGradientBoostingRegressor, HistGradientBoostingClassifier

try:
    from xgboost import XGBRegressor, XGBClassifier
    XGB_AVAILABLE = True
except Exception:
    XGB_AVAILABLE = False

try:
    from lightgbm import LGBMRegressor, LGBMClassifier
    LGBM_AVAILABLE = True
except Exception:
    LGBM_AVAILABLE = False

try:
    import optuna
    OPTUNA_AVAILABLE = True
except Exception:
    OPTUNA_AVAILABLE = False

from feature_utils import (
    set_global_seed,
    standardize_train_valid,
    monotone_log1p,
    one_hot_basis,
    polynomial_basis,
    add_interaction_columns,
    target_encode_oof,
    conditional_expectation_feature,
)
from cv_utils import (
    empirical_risk,
    make_splitter,
    oof_cv_predictions,
    cv_bias_variance_decomposition,
    leakage_inflation,
    simulate_public_private_variance,
)
from ensemble_utils import (
    blend_predictions,
    bagging_variance_formula,
    ensemble_variance_from_covariance,
    oof_stacking,
    stacking_predict,
)
from experiment_logger import ExperimentLogger, ExperimentRecord

SEED = 42
set_global_seed(SEED)

MODULE_DIR = Path('.').resolve()

## Synthetic Experiment

In [ ]:
from sklearn.datasets import make_regression

X, y = make_regression(n_samples=4000, n_features=10, noise=20.0, random_state=SEED)
rng = np.random.default_rng(SEED)

# Create explicit leakage feature.
leak = y + rng.normal(0, 5.0, size=len(y))
X_leaky = np.column_stack([X, leak])

X_train, X_test, y_train, y_test = train_test_split(X_leaky, y, test_size=0.3, random_state=SEED)
model = LinearRegression().fit(X_train, y_train)
pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, pred))
print({"rmse_with_leak": rmse})

## Real Dataset Experiment

In [ ]:
# Hybrid policy: try California Housing, fallback to Diabetes if unavailable.
try:
    cal = fetch_california_housing(as_frame=True)
    X_real = cal.data.copy()
    y_real = cal.target.to_numpy(dtype=float)
except Exception:
    diab = load_diabetes(as_frame=True)
    X_real = diab.data.copy()
    y_real = diab.target.to_numpy(dtype=float)

X_tr, X_te, y_tr, y_te = train_test_split(X_real, y_real, test_size=0.35, random_state=SEED)

# Adversarial validation: classify split origin.
adv_X = np.vstack([X_tr.values, X_te.values])
adv_y = np.concatenate([np.zeros(len(X_tr)), np.ones(len(X_te))])
adv_model = HistGradientBoostingClassifier(random_state=SEED).fit(adv_X, adv_y)
adv_auc = roc_auc_score(adv_y, adv_model.predict_proba(adv_X)[:, 1])
print({"adversarial_auc_split_detection": adv_auc})

## Diagnostic Analysis

In [ ]:
def empirical_kl(p_vals, q_vals, bins=50):
    p_hist, edges = np.histogram(p_vals, bins=bins, density=True)
    q_hist, _ = np.histogram(q_vals, bins=edges, density=True)
    eps = 1e-10
    p = np.clip(p_hist, eps, None); q = np.clip(q_hist, eps, None)
    p = p / p.sum(); q = q / q.sum()
    return float(np.sum(p * np.log(p / q)))

col = X_tr.columns[0]
kl = empirical_kl(X_tr[col].to_numpy(), X_te[col].to_numpy(), bins=40)

def ks_statistic(a, b):
    a = np.sort(np.asarray(a)); b = np.sort(np.asarray(b))
    grid = np.sort(np.unique(np.concatenate([a, b])))
    Fa = np.searchsorted(a, grid, side="right") / len(a)
    Fb = np.searchsorted(b, grid, side="right") / len(b)
    return float(np.max(np.abs(Fa - Fb)))

ks = ks_statistic(X_tr[col].to_numpy(), X_te[col].to_numpy())
print({"feature": col, "KL": kl, "KS": ks})

## Failure Analysis

In [ ]:
# Failure case: random split hides temporal/group leakage structure.
perm = np.random.default_rng(SEED).permutation(len(y_real))
cut = int(0.7 * len(perm))
idx_train, idx_test = perm[:cut], perm[cut:]
X_bad_train, X_bad_test = X_real.iloc[idx_train], X_real.iloc[idx_test]
y_bad_train, y_bad_test = y_real[idx_train], y_real[idx_test]

bad_model = HistGradientBoostingRegressor(random_state=SEED).fit(X_bad_train, y_bad_train)
bad_rmse = np.sqrt(mean_squared_error(y_bad_test, bad_model.predict(X_bad_test)))
print({"random_split_rmse_may_hide_shift": bad_rmse})

## Exercise Ladder (basic → advanced → research-level)
1. Derive non-negativity of KL using Jensen's inequality.
2. Implement multidimensional shift detection via MMD.
3. Build a leakage checklist for timestamped features.
4. Compare adversarial AUC before/after feature sanitization.

## Summary of Mathematical Insights
Leakage detection combines theory (KL/KS/information flow) and diagnostics (adversarial validation) to reject deceptively strong but non-deployable pipelines.